## 1. 環境設定 (Setup)

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
from factor_analyzer.factor_analyzer import calculate_kmo

%load_ext autoreload
%autoreload 2

# Data Simulation

In [32]:
def simulate_efa_features(
    N=3000,                 # 新聞篇數
    K=5,                    # 潛在因子數 (縮減為 5)
    feature_names=None,     # features 名稱
    make_share=True,        # 比例型資料
    noise_sd=0.6,           # 雜訊
    seed=42
):
    rng = np.random.default_rng(seed)

    # 定義因子結構字典 {Factor_Index: [Feature_List]}
    # 0: Sentiment (情緒：正向、負向、強弱語氣)
    # 1: Risk & Uncertainty (風險：不確定性、法律、地緣政治)
    # 2: ESG (ESG：環境、社會、治理)
    # 3: Macro & External (總經：通膨、利率、供應鏈、政治)
    # 4: Corporate Fundamentals (公司基本面：獲利、成長、創新、併購)
    factor_structure = {
        0: ["pos", "neg", "modal_strong", "modal_weak"], 
        1: ["uncertainty", "litigious", "constraining", "risk", "conflict"],
        2: ["esg", "environment", "social", "governance", "carbon", "diversity"], 
        3: ["inflation", "interest_rate", "forex", "political", "supply_chain", "oil_price"], 
        4: ["profitability", "growth", "liquidity", "debt", "innovation", "tech", "merger_acquisition"] 
    }

    # 若未指定 feature_names，從結構中生成
    if feature_names is None:
        feature_names = []
        for feats in factor_structure.values():
            feature_names.extend(feats)
        # 去除重複並保持順序
        feature_names = list(dict.fromkeys(feature_names))
    
    P = len(feature_names)
    feat_idx = {name: i for i, name in enumerate(feature_names)}
    
    # 1) 生成潛在因子 F (N x K)
    F = rng.normal(0, 1, size=(N, K))
    
    # 2) 生成 Loading Matrix L (P x K)
    L = np.zeros((P, K))
    
    for k, subjects in factor_structure.items():
        if k >= K: continue
        for subj in subjects:
            if subj in feat_idx:
                # 主負荷量 (Main Loading)
                L[feat_idx[subj], k] = rng.uniform(0.65, 0.95)
                
                # 隨機添加一些 Cross-loading (讓資料不要太完美)
                if rng.random() < 0.15: # 15% 機率
                    other_k = rng.integers(0, K)
                    if other_k != k:
                        L[feat_idx[subj], other_k] = rng.uniform(0.2, 0.4)

    # 手動添加一些邏輯上的 Cross-loadings
    if "neg" in feat_idx and 1 < K: 
        L[feat_idx["neg"], 1] += 0.4 # Neg -> Risk
    if "debt" in feat_idx and 1 < K:
        L[feat_idx["debt"], 1] += 0.5 # Debt -> Risk
    if "inflation" in feat_idx and 1 < K:
        L[feat_idx["inflation"], 1] += 0.3 # Inflation -> Risk
    if "supply_chain" in feat_idx and 4 < K:
        L[feat_idx["supply_chain"], 4] += 0.3 # Supply Chain -> Corporate

    # 3) 生成 features: X = F @ L.T + noise
    X = F @ L.T + rng.normal(0, noise_sd, size=(N, P))
    
    # 改進：與其直接切除負值 (ReLU)，不如加上一個基底強度 (Base Intensity)
    # 這樣可以保持線性結構，讓因子分析更精準，同時確保數值非負。
    # 這裡加上 abs(min) + 緩衝，確保所有值 > 0
    X = X + np.abs(X.min()) + 0.1
    
    # 4) 轉為比例 (Share)
    if make_share:
        row_sum = X.sum(axis=1, keepdims=True)
        # 避免除以 0
        X = X / np.clip(row_sum, 1e-8, None)
        
    df_X = pd.DataFrame(X, columns=feature_names)
    return df_X, L

In [33]:
# 例：產生 EFA 用資料 (Data Loading)
df_X, L_true = simulate_efa_features(
    N=3000,
    K=8,             # 指定 8 個因子
    make_share=False,
    noise_sd=0.6     # 雜訊設小一點，讓結構更清晰
)

print(f"資料筆數: {len(df_X)}")
print(f"變數欄位: {df_X.columns.tolist()}")
df_X

資料筆數: 3000
變數欄位: ['pos', 'neg', 'modal_strong', 'modal_weak', 'uncertainty', 'litigious', 'constraining', 'risk', 'conflict', 'esg', 'environment', 'social', 'governance', 'carbon', 'diversity', 'inflation', 'interest_rate', 'forex', 'political', 'supply_chain', 'oil_price', 'profitability', 'growth', 'liquidity', 'debt', 'innovation', 'tech', 'merger_acquisition']


,pos,neg,modal_strong,modal_weak,uncertainty,litigious,constraining,risk,conflict,esg,...,political,supply_chain,oil_price,profitability,growth,liquidity,debt,innovation,tech,merger_acquisition
0,6.124515,4.539458,4.335471,5.011473,3.219968,3.793905,3.023229,3.270114,3.699508,5.989906,...,5.593556,4.816512,6.150374,4.082558,2.983914,1.768619,2.096926,2.745661,3.663150,3.813984
1,4.037012,3.858639,3.764643,4.736470,3.971354,4.563747,4.331850,4.275464,3.153082,5.053493,...,6.323610,4.594276,6.132245,4.329483,4.808474,4.832254,4.038946,4.256349,4.495776,4.105629
2,6.088833,4.307393,5.686178,5.523860,4.133118,3.937335,4.071273,4.158664,4.065544,4.488418,...,4.973715,6.223718,4.705972,5.346173,3.643742,5.104312,4.169354,4.697617,4.694570,5.557478
3,4.535471,3.692132,3.920905,4.877448,4.659327,3.510535,4.714197,3.704726,4.686824,4.645015,...,4.671503,5.867401,4.466518,4.945069,5.265470,5.473075,4.070551,4.439161,5.351573,4.743430
4,3.488281,3.864179,4.145354,4.438536,4.346624,4.396207,3.601819,3.697717,4.662050,4.874982,...,4.569517,5.585868,5.975549,4.699633,4.326491,4.767098,4.787321,4.521474,3.235686,5.273387
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,7.077138,6.523607,6.861195,5.546204,4.009453,5.379242,4.620285,6.600762,4.745943,5.312265,...,4.023896,5.430425,3.712630,4.806303,3.852497,3.697946,4.648889,5.073187,4.717244,5.898819
2996,4.279068,4.669207,4.681700,4.744373,3.435074,3.687896,3.777073,2.946851,4.541287,6.039314,...,4.774551,5.039976,3.802812,4.753761,4.920640,5.719060,3.810217,5.886290,6.314536,4.855838
2997,3.686227,4.447862,3.310046,3.519049,6.073632,6.162549,5.153220,5.288896,5.710878,2.876220,...,6.068754,5.799174,6.422789,4.483802,6.386443,4.363508,5.909427,3.489262,4.717159,4.930491
2998,4.401310,4.154422,4.558598,4.090426,5.768169,5.153685,4.087017,5.172592,4.837929,3.908636,...,3.530632,3.993999,4.636608,4.830717,4.019970,5.678476,4.825488,5.439686,4.602976,5.222923


# 探索性因子分析 (Exploratory Factor Analysis, EFA)

主要流程：
1. **資料檢定**：檢查資料是否適合做因子分析 (Bartlett, KMO)。
2. **因子萃取**：決定最佳因子個數 (Scree Plot)。
3. **因子旋轉與解釋**：計算因子負荷量 (Factor Loadings) 以解釋因子意義。

## 1. 資料適動性檢定 (Data Suitability Tests)

*   **Bartlett’s Test of Sphericity**: 檢定變數之間是否互相獨立。H0: 變數間無相關（不適合做 EFA）。若 p-value < 0.05，則拒絕 H0，表示適合。
*   **KMO (Kaiser-Meyer-Olkin) Test**: 檢定變數間的偏相關性是否夠小。值介於 0~1，通常 > 0.6 表示適合。

In [34]:
# 1. Bartlett's Test
chi_square_value, p_value = calculate_bartlett_sphericity(df_X)
print(f"Bartlett's Test p-value: {p_value:.4e}")

# 2. KMO Test
kmo_all, kmo_model = calculate_kmo(df_X)
print(f"KMO Test Value: {kmo_model:.4f}")

if p_value < 0.05 and kmo_model > 0.6:
    print("=> 資料適合進行因子分析")
else:
    print("=> 資料可能不適合進行因子分析，請檢查變數相關性")

Bartlett's Test p-value: 0.0000e+00
KMO Test Value: 0.9135
=> 資料適合進行因子分析


c:\Users\ownme\anaconda3\envs\ndhu_windows\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning:

The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.



## 2. 決定因子個數 (Factor Selection)

使用 **陡坡圖 (Scree Plot)** 與 **特徵值 (Eigenvalue) > 1** 準則來輔助判斷。

In [35]:
fa = FactorAnalyzer(n_factors=len(df_X.columns), rotation=None)
fa.fit(df_X)

# 取得特徵值
ev, v = fa.get_eigenvalues()

# 使用 Plotly 繪製 Steep Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, df_X.shape[1]+1)),
    y=ev,
    mode='lines+markers',
    name='Eigenvalues'
))

# 加入 Eigenvalue = 1 的參考線
fig.add_hline(y=1, line_dash="dash", line_color="red", annotation_text="Eigenvalue=1")

fig.update_layout(
    title='Scree Plot (陡坡圖)',
    xaxis_title='Factors',
    yaxis_title='Eigenvalue',
    template='plotly_white'
)
fig.show()

n_factors_ev1 = sum(ev > 1)
print(f"特徵值 > 1 的因子個數: {n_factors_ev1}")
print(f"各因子特徵值: {np.round(ev, 2)}")

c:\Users\ownme\anaconda3\envs\ndhu_windows\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



特徵值 > 1 的因子個數: 5
各因子特徵值: [4.95 4.29 3.91 3.65 2.65 0.49 0.47 0.46 0.45 0.44 0.43 0.42 0.41 0.39
 0.39 0.38 0.37 0.36 0.35 0.34 0.33 0.32 0.32 0.31 0.3  0.29 0.28 0.27]


### 補充：平行分析 (Parallel Analysis)

單純使用特徵值 > 1 (Kaiser Criterion) 有時會高估因子個數。
**平行分析**透過模擬相同維度的隨機資料，計算隨機相關矩陣的特徵值分佈。
準則：若 **真實數據特徵值 > 隨機數據特徵值的 95% 分位數**，則保留該因子。

In [36]:
# 平行分析 (Parallel Analysis) 實作
def run_parallel_analysis(df, n_iter=100, seed=42):
    n, p = df.shape
    rng = np.random.default_rng(seed)
    random_eigenvalues = []
    
    print(f"開始進行平行分析 (模擬 {n_iter} 次)...")
    for _ in range(n_iter):
        # 產生隨機矩陣 (常態分佈)
        random_data = rng.normal(0, 1, size=(n, p))
        # 計算相關係數矩陣
        corr_matrix = np.corrcoef(random_data, rowvar=False)
        # 計算特徵值
        evs = np.linalg.eigvalsh(corr_matrix)
        # 排序 (大 -> 小)
        random_eigenvalues.append(evs[::-1])
        
    # 計算 95% 分位數
    random_eigenvalues = np.array(random_eigenvalues)
    percentile_95 = np.percentile(random_eigenvalues, 95, axis=0)
    
    return percentile_95

# 執行分析
random_ev_95 = run_parallel_analysis(df_X)

# 取得真實特徵值 (沿用上個 cell 的結果)
real_ev, _ = fa.get_eigenvalues()

# 決定因子個數
n_factors_pa = sum(real_ev > random_ev_95)

# 繪圖
fig = go.Figure()

# 1. 真實特徵值
fig.add_trace(go.Scatter(
    x=list(range(1, len(real_ev)+1)),
    y=real_ev,
    mode='lines+markers',
    name='Actual Eigenvalues',
    line=dict(color='blue')
))

# 2. 隨機特徵值 (95% CI)
fig.add_trace(go.Scatter(
    x=list(range(1, len(random_ev_95)+1)),
    y=random_ev_95,
    mode='lines',
    name='Random Data (95th %ile)',
    line=dict(color='red', dash='dash')
))

fig.update_layout(
    title='Parallel Analysis (平行分析)',
    xaxis_title='Factors',
    yaxis_title='Eigenvalue',
    template='plotly_white',
    legend=dict(x=0.7, y=0.9)
)
fig.show()

print(f"Parallel Analysis 建議保留因子數: {n_factors_pa}")
print(f"特徵值 > 1 建議保留因子數: {sum(real_ev > 1)}")

開始進行平行分析 (模擬 100 次)...


Parallel Analysis 建議保留因子數: 5
特徵值 > 1 建議保留因子數: 5


## 3. 執行因子分析與視覺化 (Factor Extraction & Visualization)

*   **Rotation**: 使用 `promax` (非正交旋轉)，因為新聞語意特徵（如風險、政策）通常具有相關性。
*   **Factors**: 設定為 4 (模擬設定是 4，通常由 Scree Plot 判斷)。

In [37]:
# 設定因子數，並使用 promax 旋轉
fa = FactorAnalyzer(n_factors=5, rotation='promax')
fa.fit(df_X)

# 取得因子負荷量 (Loading Matrix)
loadings = pd.DataFrame(
    fa.loadings_, 
    index=df_X.columns, 
    columns=[f'Factor{i+1}' for i in range(fa.loadings_.shape[1])]
)

# 使用 Plotly 繪製 Heatmap
fig = px.imshow(
    loadings,
    x=loadings.columns,
    y=loadings.index,
    text_auto='.2f',          # 顯示數值 (小數點2位)
    aspect="auto",            # 自動調整長寬比，關鍵參數！
    color_continuous_scale='RdBu_r',  # 紅藍配色，紅色代表高負荷
    origin='upper',
    title='Factor Loadings (Expanded View)'
)
fig.update_layout(
    height=1000,              # 拉長高度 (視變數數量調整，例如 800~1200)
    width=600,                # 設定寬度
    autosize=False
)
fig.show()

print("因子負荷量表:")
loadings

c:\Users\ownme\anaconda3\envs\ndhu_windows\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



因子負荷量表:


,Factor1,Factor2,Factor3,Factor4,Factor5
pos,-0.027620,-0.009587,-0.003265,-0.066627,0.731501
neg,0.004535,-0.006136,-0.003623,0.288844,0.756486
modal_strong,-0.002824,-0.001165,0.015357,-0.087153,0.829575
modal_weak,0.017404,0.009786,-0.005576,-0.067668,0.735429
uncertainty,-0.020769,0.019044,0.016276,0.795347,-0.017655
litigious,-0.024527,-0.013570,-0.016323,0.742608,0.006303
constraining,-0.018596,0.006884,-0.019262,0.779975,-0.010773
risk,-0.022777,0.000412,-0.014278,0.825230,-0.011807
conflict,-0.010901,0.004435,0.012081,0.766234,-0.018396
esg,-0.053448,0.825417,-0.001206,-0.005725,-0.004932


### 解釋變異量 (Factor Variance)
檢視每個因子解釋了多少資料變異，以及累計解釋變異量。

In [38]:
# 1. 取得資料 (Tuple)
variance_tuple = fa.get_factor_variance()

# 2. 轉成 Numpy Array 以便處理形狀
variance_arr = np.array(variance_tuple)

# 3. 建立 DataFrame
variance_df = pd.DataFrame(
    variance_arr, 
    index=['SS Loadings', 'Proportion Var', 'Cumulative Var'],
    columns=[f'Factor{i+1}' for i in range(variance_arr.shape[1])]
)

variance_df

,Factor1,Factor2,Factor3,Factor4,Factor5
SS Loadings,4.254606,3.869052,3.743913,3.404391,2.340413
Proportion Var,0.151950,0.138180,0.133711,0.121585,0.083586
Cumulative Var,0.151950,0.290131,0.423842,0.545427,0.629013


### 3.1 共同性檢定 (Communalities)

檢視每個變數變異量中，能被抽取出的因子解釋的比例。若數值過低 (< 0.2 或 0.3)，表示該變數與提取出的因子關聯性弱，可考慮移除。

In [48]:
# 取得共同性 (Communalities)
communalities = fa.get_communalities()
comm_df = pd.DataFrame(communalities, index=df_X.columns, columns=['Communalities'])

print("共同性 (Communalities) - 前 5 筆:")
print(comm_df)

# 檢視是否有共同性過低的變數 (< 0.2 或 0.3)
low_comm_vars = comm_df[comm_df['Communalities'] < 0.3]
if not low_comm_vars.empty:
    print("\n[Warning] 以下變數共同性較低 (< 0.3)，可能需考慮移除：")
    print(low_comm_vars)
else:
    print("\n所有變數共同性皆 > 0.3，表現良好。")

# 繪圖檢視
fig = px.bar(comm_df, x=comm_df.index, y='Communalities', title='Variable Communalities')
fig.add_hline(y=0.3, line_dash="dash", line_color="red", annotation_text="Threshold 0.3")
fig.show()

共同性 (Communalities) - 前 5 筆:
                    Communalities
pos                      0.540398
neg                      0.655773
modal_strong             0.696035
modal_weak               0.545864
uncertainty              0.633948
litigious                0.552559
constraining             0.609242
risk                     0.681866
conflict                 0.587738
esg                      0.684229
environment              0.724434
social                   0.583148
governance               0.541291
carbon                   0.710254
diversity                0.661512
inflation                0.707242
interest_rate            0.639085
forex                    0.588539
political                0.633634
supply_chain             0.697745
oil_price                0.653348
profitability            0.580956
growth                   0.555945
liquidity                0.700448
debt                     0.675369
innovation               0.580528
tech                     0.608189
merger_acquisition 

### 3.2 因子相關矩陣 (Factor Correlation Matrix)

由於使用 Promax 斜交旋轉，因子之間允許存在相關性。此矩陣可顯示因子間的關聯程度。

In [40]:
# 因子相關矩陣
if fa.rotation == 'promax' or fa.rotation == 'oblimin':
    try:
        # 對於 Oblique Rotation，phi_ 即為因子相關矩陣
        factor_corr = pd.DataFrame(
            fa.phi_,
            index=[f'Factor{i+1}' for i in range(fa.phi_.shape[1])],
            columns=[f'Factor{i+1}' for i in range(fa.phi_.shape[1])]
        )
        
        print("因子相關矩陣:")
        print(factor_corr)
        
        # 繪製 Heatmap
        fig = px.imshow(
            factor_corr, 
            text_auto='.2f',
            color_continuous_scale='RdBu_r', 
            zmin=-1, zmax=1,
            title='Factor Correlation Matrix'
        )
        fig.show()
    except AttributeError:
        print("無法取得因子相關矩陣 (phi_)，可能因版本差異或旋轉未成功。")
else:
    print("目前使用正交旋轉 (Orthogonal Rotation)，因子間假設為獨立 (相關性為 0)。")

因子相關矩陣:
          Factor1   Factor2   Factor3   Factor4   Factor5
Factor1  1.000000  0.050164  0.069301  0.072298  0.011423
Factor2  0.050164  1.000000  0.089380 -0.019027 -0.015038
Factor3  0.069301  0.089380  1.000000  0.044350  0.024546
Factor4  0.072298 -0.019027  0.044350  1.000000  0.103649
Factor5  0.011423 -0.015038  0.024546  0.103649  1.000000


## 4. 因子命名與解釋 (Factor Interpretation)

列出每個因子中，負荷量 (Loading) 絕對值大於 0.5 的變數，以輔助命名。

In [41]:
def get_factor_loadings_report(loadings_df, threshold=0.5):
    factors = loadings_df.columns
    report = {}
    for factor in factors:
        # 篩選出負荷量絕對值 > threshold 的變數
        high_loading_vars = loadings_df[factor][abs(loadings_df[factor]) > threshold]
        # 依絕對值大小排序
        high_loading_vars = high_loading_vars.reindex(
            high_loading_vars.abs().sort_values(ascending=False).index
        )
        report[factor] = high_loading_vars
        
    return report

loading_report = get_factor_loadings_report(loadings, threshold=0.5)

for factor, vars_series in loading_report.items():
    print(f"=== {factor} ===")
    if vars_series.empty:
        print("  (No variables > threshold)")
    else:
        for var_name, loading_val in vars_series.items():
            print(f"  {var_name}: {loading_val:.3f}")
    print()

=== Factor1 ===
  liquidity: 0.835
  tech: 0.778
  merger_acquisition: 0.762
  profitability: 0.761
  innovation: 0.760
  growth: 0.744
  debt: 0.712

=== Factor2 ===
  environment: 0.850
  carbon: 0.842
  esg: 0.825
  diversity: 0.812
  governance: 0.734
  social: 0.711

=== Factor3 ===
  inflation: 0.804
  oil_price: 0.804
  supply_chain: 0.800
  interest_rate: 0.797
  political: 0.766
  forex: 0.765

=== Factor4 ===
  risk: 0.825
  uncertainty: 0.795
  constraining: 0.780
  conflict: 0.766
  litigious: 0.743

=== Factor5 ===
  modal_strong: 0.830
  neg: 0.756
  modal_weak: 0.735
  pos: 0.732



## 4. 因子定義與命名 (Factor Interpretation)

根據上述 Loading Table 與模擬資料設定，我們將 5 個潛在因子定義如下。這有助於後續對因子分數的解釋與應用。

| 因子 (Factor) | 命名 (Name) | 主要特徵 (Key Features) |
| :--- | :--- | :--- |
| **Factor 1** | **情緒 (Sentiment)** | 正負面情緒、語氣強弱 (pos, neg, modal_strong...) |
| **Factor 2** | **風險 (Risk)** | 不確定性、法律風險、衝突 (uncertainty, risk, litigious...) |
| **Factor 3** | **ESG** | 環境、社會、治理、碳排放 (esg, environment, governance...) |
| **Factor 4** | **總體經濟 (Macro)** | 通膨、利率、匯率、供應鏈 (inflation, interest_rate, supply_chain...) |
| **Factor 5** | **公司基本面 (Fundamentals)** | 獲利、成長、債務、創新 (profitability, growth, tech...) |

> **註**：實際 EFA 萃取出的因子順序可能與上述略有不同（取決於解釋變異量排序），請參照 Loadings Heatmap 進行對應。

## 5. 信度分析 (Reliability Analysis)

使用 **Cronbach's Alpha** 檢驗各因子內部一致性 (Internal Consistency)。
*   $\alpha > 0.7$: 信度可接受
*   $\alpha > 0.8$: 信度良好

In [42]:
def cronbach_alpha(df):
    # 1. 計算變數數 k
    k = df.shape[1]
    if k < 2:
        return np.nan
    
    # 2. 計算各變數變異數總和
    sum_var_items = df.var(ddof=1).sum()
    
    # 3. 計算總分變異數
    var_total = df.sum(axis=1).var(ddof=1)
    
    # 4. 公式
    alpha = (k / (k - 1)) * (1 - (sum_var_items / var_total))
    return alpha

print("Cronbach's Alpha per Factor:")
for factor in loading_report.keys():
    # 取出該因子下的變數名稱
    items = loading_report[factor].index.tolist()
    
    if len(items) > 1:
        # 從原始資料 df_X 中取出這些欄位
        factor_df = df_X[items]
        alpha = cronbach_alpha(factor_df)
        print(f"  {factor} (items={len(items)}): {alpha:.4f}")
    else:
        print(f"  {factor}: Item count < 2, cannot calculate Alpha")

Cronbach's Alpha per Factor:
  Factor1 (items=7): 0.9055
  Factor2 (items=6): 0.9111
  Factor3 (items=6): 0.9057
  Factor4 (items=5): 0.8851
  Factor5 (items=4): 0.8422


## 6. 因子分數萃取 (Factor Scores Extraction)

計算每筆資料在各個因子上的得分，以便進行後續分析 (Regression, Clustering etc.)。

In [47]:
# 計算因子分數
factor_scores = fa.transform(df_X)

# 轉成 DataFrame
df_scores = pd.DataFrame(
    factor_scores, 
    columns=[f'Score_{i+1}' for i in range(factor_scores.shape[1])]
)

# 合併回原始資料 (Optional)
df_full = pd.concat([df_X.reset_index(drop=True), df_scores], axis=1)

df_scores

c:\Users\ownme\anaconda3\envs\ndhu_windows\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



,Score_1,Score_2,Score_3,Score_4,Score_5
0,-1.984662,0.844051,0.782927,-1.601278,0.303480
1,-0.332501,0.753996,0.962416,-0.710733,-0.663338
2,0.307018,0.297582,0.256353,-0.843328,0.794710
3,0.311450,0.605258,0.892537,-0.516446,-0.534908
4,-0.148141,0.104795,0.666994,-0.610435,-0.732740
...,...,...,...,...,...
2995,0.024638,0.641854,-0.085448,0.433430,2.073533
2996,0.701274,0.855554,-0.121255,-1.221046,-0.056368
2997,0.215165,-1.745608,1.008040,1.190587,-1.076597
2998,0.309974,0.521452,-0.509583,0.328221,-0.417988


## 7. 應用：預測大盤漲跌 (Application: Stock Market Prediction)

假設我們想驗證這些「新聞情緒因子」是否能預測股市。我們建立一個模擬的預測任務：
1. **建立 Target ($y$)**：模擬大盤漲跌 (1=漲, 0=跌)。
2. **特徵 ($X$)**：使用前面萃取出的 5 個因子分數。
3. **模型**：Logistic Regression。
4. **解釋**：觀察哪些因子對股市影響最大。

In [44]:
# 1. 模擬大盤漲跌 (Target Generation)
# 假設邏輯：
#   - Sentiment (F1) 上升 -> 漲 (+)
#   - Risk (F2) 上升 -> 跌 (-)
#   - ESG (F3) -> 微幅正向 (+)
#   - Macro (F4) -> 負向 (通膨高對股市不好) (-)
#   - Fundamentals (F5) -> 正向 (+)

np.random.seed(42)

# 產生合成 Logits
# 係數設定: F1(+0.8), F2(-0.9), F3(+0.2), F4(-0.5), F5(+0.6)
true_coefs = np.array([0.8, -0.9, 0.2, -0.5, 0.6])
logits = df_scores.values @ true_coefs + np.random.normal(0, 1, size=len(df_scores))

# 轉成機率與類別 (Sigmoid)
probs = 1 / (1 + np.exp(-logits))
y = (probs > 0.5).astype(int)

df_model = df_scores.copy()
df_model['Target_Return'] = y

print(f"上漲天數比例: {y.mean():.2%}")
df_model.head()

上漲天數比例: 50.87%


,Score_1,Score_2,Score_3,Score_4,Score_5,Target_Return
0,-1.984662,0.844051,0.782927,-1.601278,0.303480,0
1,-0.332501,0.753996,0.962416,-0.710733,-0.663338,0
2,0.307018,0.297582,0.256353,-0.843328,0.794710,1
3,0.311450,0.605258,0.892537,-0.516446,-0.534908,1
4,-0.148141,0.104795,0.666994,-0.610435,-0.732740,0


### 7.1 模型訓練與評估 (Model Training & Evaluation)

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 準備資料
X = df_model.drop(columns=['Target_Return'])
y = df_model['Target_Return']

# 切分訓練/測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 訓練模型
model = LogisticRegression()
model.fit(X_train, y_train)

# 預測
y_pred = model.predict(X_test)

# 評估
acc = accuracy_score(y_test, y_pred)
print(f"預測準確率 (Accuracy): {acc:.2%}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

預測準確率 (Accuracy): 80.56%

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.79      0.80       438
           1       0.81      0.82      0.81       462

    accuracy                           0.81       900
   macro avg       0.81      0.81      0.81       900
weighted avg       0.81      0.81      0.81       900



### 7.2 因子影響力視覺化 (Factor Importance)

In [46]:
# 提取係數
coef_df = pd.DataFrame({
    'Factor': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=True)

# 繪圖
fig = px.bar(
    coef_df, 
    x='Coefficient', 
    y='Factor', 
    orientation='h',
    title='因子對大盤漲跌的影響力 (Logistic Regression coefficients)',
    color='Coefficient',
    color_continuous_scale='RdBu'
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.show()